<a href="https://colab.research.google.com/github/sujithkumarmp/cloud-ai-basics/blob/main/hf_tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import transformers

In [3]:
torch.__version__

'2.10.0+cpu'

In [5]:
from transformers import AutoTokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
sentence =" What is the capital of France?"

In [8]:
input_ids = tokenizer(sentence, return_tensors="pt").input_ids

Token ids

In [9]:
input_ids

tensor([[1867,  318,  262, 3139,  286, 4881,   30]])

In [12]:
tokenizer.decode(286)

' of'

In [14]:
for token_id in input_ids[0]:
  print(tokenizer.decode(token_id))


 What
 is
 the
 capital
 of
 France
?


In [16]:
from transformers import AutoModelForCausalLM

In [17]:
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [18]:
input_ids

tensor([[1867,  318,  262, 3139,  286, 4881,   30]])

In [19]:
output = gpt2(input_ids)

1. batch, 2. num of tokens and 3. Unique tokens

In [21]:
output.logits.shape

tensor([[[ -37.9883,  -37.9579,  -41.4890,  ...,  -44.0717,  -43.7975,
           -37.7046],
         [ -90.5062,  -88.9512,  -95.2817,  ...,  -95.2052,  -95.9588,
           -90.8715],
         [ -96.4428,  -94.5894,  -97.7109,  ...,  -96.9249, -100.0142,
           -94.3136],
         ...,
         [ -75.6492,  -74.3697,  -78.5434,  ...,  -80.8501,  -82.3028,
           -77.1403],
         [ -84.8356,  -85.3072,  -89.1383,  ...,  -96.8418,  -94.1887,
           -87.7884],
         [-128.5583, -128.8332, -129.8123,  ..., -139.8500, -139.4842,
          -125.3359]]], grad_fn=<UnsafeViewBackward0>)

In [22]:
final_logits = gpt2(input_ids).logits[0,-1]

In [24]:
final_logits

tensor([-128.5583, -128.8332, -129.8123,  ..., -139.8500, -139.4842,
        -125.3359], grad_fn=<SelectBackward0>)

In [25]:
final_logits.argmax()  # token id

tensor(198)

In [27]:
tokenizer.decode(final_logits.argmax())

'\n'

In [29]:
top_10 = torch.topk(final_logits, 10)
for index in top_10.indices:
  print(tokenizer.decode(index))




 The
 It
 What
 And
 France
 Is
 How
 I
 In


In [30]:
final_logits.softmax(dim=0)

tensor([1.2559e-04, 9.5409e-05, 3.5838e-05,  ..., 1.5669e-09, 2.2590e-09,
        3.1508e-03], grad_fn=<SoftmaxBackward0>)

In [37]:
top10 = torch.topk(final_logits.softmax(dim=0),10)
for value, index in zip(top10.values, top10.indices):
  print(f"{tokenizer.decode(index)} -- {value.item():.1%}")


 -- 26.9%
 The -- 5.5%
 It -- 4.2%
 What -- 3.4%
 And -- 2.9%
 France -- 2.3%
 Is -- 2.0%
 How -- 1.9%
 I -- 1.5%
 In -- 1.2%


Text generation

In [55]:
output_ids = gpt2.generate(input_ids,max_new_tokens=20, do_sample=False, top_k=4, top_p=.95)

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [56]:
decoded_text = tokenizer.decode(output_ids[0])

In [57]:
decoded_text

' What is the capital of France?\n\nThe capital of France is Paris. It is the capital of France. It is the capital'